# Train Startup Agent with Unsloth & TRL

This notebook demonstrates how to train an LLM to play the **Startup Survival Simulator** using [Unsloth](https://github.com/unslothai/unsloth) and Hugging Face `trl`.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

max_seq_length = 2048
dtype = None
load_in_4bit = True

# 1. Load Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 2. Inject LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",    
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## Dataset Preparation
We use generated high-reward trajectories from the environment. Each entry is formatted as a system/user/assistant interaction.

In [ ]:
# Example synthetic offline data mapping states to optimal actions
data = [
    {"instruction": "You are an AI CEO. Current objective: survival.", 
     "input": "Current startup state: {\"cash\": 50000.0, \"users\": 100, \"revenue\": 1000.0, \"burn_rate\": 4500.0, \"product_quality\": 0.55, \"market_demand\": 0.6, \"morale\": 0.7}",
     "output": "improve_product"},
    {"instruction": "You are an AI CEO. Current objective: survival.", 
     "input": "Current startup state: {\"cash\": 15000.0, \"users\": 800, \"revenue\": 8500.0, \"burn_rate\": 11000.0, \"product_quality\": 0.65, \"market_demand\": 0.8, \"morale\": 0.5}",
     "output": "reduce_costs"},
]

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts }

dataset = Dataset.from_list(data)
dataset = dataset.map(formatting_prompts_func, batched = True)

## Training with SFTTrainer

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# 3. Save Model
model.save_pretrained("startup_agent_lora")
tokenizer.save_pretrained("startup_agent_lora")
print("Training complete! LoRA adapters saved.")